In [4]:
"""The main class that supplies other files"""
import kagglehub
import os
import joblib
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import f1_score
from sklearn.preprocessing import FunctionTransformer

In [8]:
import kagglehub
cwd=os.getcwd()
os.chdir(cwd)
# Download latest version
path = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")
print(f'path to download, {path}')

In [4]:
!pip install -U spacy --quiet
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.1 MB/s  0:00:00.9 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
df= pd.read_csv(f'{path}/emails.csv')

In [6]:
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [8]:
df.text.str.len().sum()

np.int64(8917171)

In [9]:
nlp=spacy.load('en_core_web_sm') # Function to clean and process the data for training
def extract(x):
    df=pd.DataFrame(x,columns=['text'])
    def lemma(text): 
        doc=nlp(''.join(text)) 
        tok=[token.lemma_ for token in doc 
             if not token.is_punct and not token.is_stop] 
        return ' '.join(tok) 
    df['text']=df['text'].str.lower().apply(lemma)
    return df.to_numpy()

In [27]:
print(df['text'].str.len().sum())
df.spam.value_counts(normalize=True)  #Checking to see how unbalanced is the dataset

8917171


spam
0    0.761173
1    0.238827
Name: proportion, dtype: float64

In [11]:
Xtrain, Xtest, ytrain, ytest=train_test_split(df['text'].values,df['spam'].values,
                                              test_size=.1,
                                              random_state=42,
                                              shuffle=True)

In [12]:
print(Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape)

(5155,) (573,) (5155,) (573,)


In [10]:
text_proc = FunctionTransformer(extract)
pipe=Pipeline([('func', text_proc), ('transformer',TfidfVectorizer()),
               ('model', LinearSVC())
              ]
             )
params = [
    {
        'model': [LinearSVC()],
        'model__C': np.linspace(0.5, 2, 6),
        'model__class_weight': [{0: 1, 1: v} for v in range(1, 6)]
    },
    {
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': np.linspace(0.5, 2, 6),
        'model__class_weight': [{0: 1, 1: v} for v in range(1, 6)]
    }
    ]
grid= GridSearchCV(pipe,
                 param_grid=params,
                 cv=5,
                 #scoring='f1',
                 refit=True,
                verbose=10
                 )

In [ ]:
mod= grid.fit(Xtrain, ytrain)

In [ ]:
mod.best_estimator_

In [ ]:
f1_score(mod.best_estimator_.predict(Xtest), ytest)

In [ ]:
t= [r"Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the"]
doc= nlp(''.join(t))
for token in doc:
    print(token.lemma_)

In [191]:
mod.best_estimator_.predict([r"""Hello Micah,We have received your application for the Data Analytics Specialist position. 
Thank you for your interest. We'll be in touch once we've been able to review your application.
In the meantime, why not learn more about us
At BT we believe in empowering our people to thrive and make a real impact."""])

array([1])

### Model depolyment

In [197]:
with open('trained_model', 'wb') as model:
    pickle.dump(mod, model)


def prediction(text):
    with open('trained_model', 'rb') as model:
        model=pickle.load(model)
        print('Spam') if model.best_estimator_.predict(text)==1 else print('Not Spam') 

In [200]:
prediction([r"""Hello Micah,We have received your application for the Data Analytics Specialist position. 
Thank you for your interest. We'll be in touch once we've been able to review your application.
In the meantime, why not learn more about us
At BT we believe in empowering our people to thrive and make a real impact."""])

Not Spam
